# RoPE — Rotary Position Embeddings

**Paper:** RoFormer: Enhanced Transformer with Rotary Position Embedding (Su et al. 2021)

**Used in:** LLaMA, LLaMA-2, Mistral, Gemma, Qwen, Falcon — basically every modern LLM.

## Why position embeddings exist

Transformers are permutation-invariant — attention has no concept of order.
If you shuffle tokens, you get the same attention scores (tokens don't know where they are).
Position embeddings inject position information.

## Absolute vs Relative vs RoPE

| Method | How | Problem |
|---|---|---|
| Learned absolute (GPT-2) | Add position vector to token embedding | Can't generalize beyond training length |
| Sinusoidal (original Transformer) | Fixed sin/cos vectors added to input | Still absolute, limited extrapolation |
| Relative (T5) | Bias the attention scores by distance | Complex, slow |
| **RoPE** | Rotate Q and K vectors by position angle | Naturally encodes relative distance, extrapolates well |

## The RoPE Idea

Instead of adding position to the embeddings, **rotate** Q and K vectors by an angle proportional to their position.

```
q at position m → rotate by angle m*θ
k at position n → rotate by angle n*θ

Dot product q·k depends on (m-n)*θ → only the RELATIVE distance matters
```

This means attention scores naturally encode relative position, not absolute.

## Resources

| Resource | Link |
|---|---|
| Original paper — RoFormer (Su et al. 2021) | https://arxiv.org/abs/2104.09864 |
| EleutherAI blog — Rotary Embeddings | https://blog.eleutherai.org/rotary-embeddings |
| LLaMA implementation reference (Meta) | https://github.com/meta-llama/llama/blob/main/llama/model.py |
| Illustrated explainer (Sebastián Raschka) | https://magazine.sebastianraschka.com/p/understanding-and-coding-self-attention |
| YaRN (extended context with RoPE) | https://arxiv.org/abs/2309.00071 |

In [ ]:
import torch
import torch.nn as nn
import math

def precompute_rope_freqs(dim, max_seq_len, base=10000):
    """
    Precompute rotation angles for each position and dimension.
    dim: head dimension (must be even)
    base: 10000 in original paper, larger values = longer context range
    """
    # Frequency for each pair of dimensions: θ_i = 1 / (base^(2i/dim))
    # dim/2 frequencies, one per pair of dimensions
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))  # [dim/2]

    # Positions
    positions = torch.arange(max_seq_len).float()   # [seq_len]

    # Outer product: each position gets each frequency
    freqs = torch.outer(positions, freqs)   # [seq_len, dim/2]

    # Convert to complex form: cos + i*sin
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)   # [seq_len, dim/2] complex
    return freqs_cis


freqs = precompute_rope_freqs(dim=64, max_seq_len=2048)
print('Freqs shape:', freqs.shape)   # [2048, 32] complex

In [ ]:
def apply_rope(x, freqs_cis):
    """
    Apply rotary embeddings to query or key tensor.
    x: [batch, seq_len, heads, head_dim]
    freqs_cis: [seq_len, head_dim/2] complex
    """
    # Reshape x to complex: pair up dimensions [d0,d1, d2,d3,...] → [(d0+id1), (d2+id3),...]
    x_complex = torch.view_as_complex(
        x.float().reshape(*x.shape[:-1], -1, 2)   # [..., head_dim/2, 2] → [..., head_dim/2] complex
    )

    # Align freqs_cis shape for broadcasting: [seq_len, head_dim/2] → [1, seq_len, 1, head_dim/2]
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(2)

    # Rotate: multiply in complex space = rotate in 2D plane
    x_rotated = x_complex * freqs_cis

    # Back to real: [(d0+id1)] → [d0, d1]
    return torch.view_as_real(x_rotated).flatten(-2).type_as(x)


# Test
batch, seq_len, heads, d_k = 2, 16, 4, 64
Q = torch.randn(batch, seq_len, heads, d_k)
freqs = precompute_rope_freqs(dim=d_k, max_seq_len=seq_len)

Q_rotated = apply_rope(Q, freqs)
print('Input shape: ', Q.shape)
print('Output shape:', Q_rotated.shape)  # same shape, just rotated
print('Values changed:', not torch.allclose(Q, Q_rotated))  # True — rotated

In [ ]:
# --- Verify the key property: dot product depends only on RELATIVE distance ---
import torch.nn.functional as F

d_k  = 64
freqs = precompute_rope_freqs(dim=d_k, max_seq_len=100)

# Create a single query and two keys at different absolute positions
# but same relative distance from the query
q_pos5  = torch.randn(1, 1, 1, d_k)
k_pos7  = torch.randn(1, 1, 1, d_k)  # relative distance = 2
k_pos12 = torch.randn(1, 1, 1, d_k)  # different key

# Make k_pos12 same vector as k_pos7 but at position 3 from a q at position 1
# (same relative distance = 2)
q_pos1  = q_pos5.clone()
k_pos3  = k_pos7.clone()

# Apply RoPE
q5_rope  = apply_rope(q_pos5,  freqs[5:6])
k7_rope  = apply_rope(k_pos7,  freqs[7:8])
q1_rope  = apply_rope(q_pos1,  freqs[1:2])
k3_rope  = apply_rope(k_pos3,  freqs[3:4])

# Dot products — both have relative distance 2
dot_5_7 = (q5_rope * k7_rope).sum().item()
dot_1_3 = (q1_rope * k3_rope).sum().item()

print(f'dot(q@pos5, k@pos7):  {dot_5_7:.6f}  (relative dist=2)')
print(f'dot(q@pos1, k@pos3):  {dot_1_3:.6f}  (relative dist=2, same vectors)')
print(f'Match (relative position encodes): {abs(dot_5_7 - dot_1_3) < 1e-4}')

## RoPE in a full attention layer

In [ ]:
class RoPEAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_len=2048):
        super().__init__()
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        # Precompute frequencies once
        freqs = precompute_rope_freqs(self.d_k, max_seq_len)
        self.register_buffer('freqs_cis', freqs)   # saved with model, not a parameter

    def forward(self, x):
        batch, seq_len, _ = x.shape

        Q = self.W_q(x).reshape(batch, seq_len, self.num_heads, self.d_k)
        K = self.W_k(x).reshape(batch, seq_len, self.num_heads, self.d_k)
        V = self.W_v(x).reshape(batch, seq_len, self.num_heads, self.d_k)

        # Apply RoPE to Q and K (NOT V — values don't need position)
        Q = apply_rope(Q, self.freqs_cis[:seq_len])
        K = apply_rope(K, self.freqs_cis[:seq_len])

        # Standard attention from here
        Q = Q.transpose(1, 2)   # [batch, heads, seq, d_k]
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        scores  = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        weights = F.softmax(scores, dim=-1)
        out     = (weights @ V).transpose(1, 2).reshape(batch, seq_len, -1)
        return self.W_o(out)


attn = RoPEAttention(d_model=256, num_heads=4)
x    = torch.randn(2, 32, 256)
out  = attn(x)
print('RoPE attention output:', out.shape)   # [2, 32, 256]